In [59]:
from nd2reader import ND2Reader
import tifffile
import numpy as np
from matplotlib import pyplot as plt
import glob
import sys
import numpy as np
import os
import importlib
from scipy.ndimage import label
import natsort
import shutil

# from file_formatting.ImageHandler import ImageHandler as ImageHandler
# os.chdir('/Users/bebr1814/projects/anabaena/scratch_data/')

In [ ]:
# # It apparently isn't possible to get the path of this notebook automatically, so set this when you need it
# sys.path.append('/Users/bebr1814/projects/anabaena/src')
# from file_formatting.ImageHandler import ImageHandler as ImageHandler

In [ ]:
# %load_ext autoreload
# %autoreload 1
# %aimport file_formatting.ImageHandler

In [51]:
# files = glob.glob('/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB00*/nd2/*.nd2')
# f = files[0]
# f

In [23]:
# # select files to convert
# files = glob.glob('/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB00*/nd2/*.nd2')

# for f in files:
# 	print(f)

# 	# make a dir
# 	maindir = os.path.dirname(os.path.dirname(f))
# 	os.makedirs(os.path.join(maindir,'tif'), exist_ok=True)
# 	outfile = os.path.join(maindir,'tif',os.path.basename(f)).replace('.nd2', '.tif')

# 	processor = ImageHandler(
# 		input_file=f,
# 		output_dir=os.path.join(maindir,'tif'),
# 		# selected_channels=selected_channels,
# 	)
# 	processor.setup_reader()
# 	processor.process_and_save_frames(
# 		save_png=False,
# 	)

# 	# ih = ImageHandler(f)
# 	# ih.read_image()
# 	# ih.save_image(outfile)
# 	# print(f'Converted {f} to {outfile}')


# Settings

In [60]:
train_path = '/Users/bebr1814/projects/anabaena/scratch_data/bulk_training_set'

In [85]:
def normalize(img):
	img = img.astype(np.float32)
	img = (img - img.min()) / (img.max() - img.min())
	img = (img * 255).astype(np.uint8)
	return img

# Movies

In [91]:
movie_mask = tifffile.imread('/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_series1_cellMask_cropped.tif')
print(movie_mask.shape)
movie_mask = np.where(movie_mask > 0, 1, 0)
for frame in range(len(movie_mask)):
	labeled_mask, num_clusters = label(np.array(movie_mask[frame], dtype=int))
	# plt.imshow(labeled_mask)
	# plt.show()
	# break
	tifffile.imwrite(f'/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_cropped.frame_{frame}.merged_mask.tiff',labeled_mask)

(80, 486, 691)


In [92]:
movie = tifffile.imread('/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_cropped.tif')
print(movie.shape)
for frame in range(movie.shape[0]):
	# save the frame's channels as one tiff file
	output_img = movie[frame,:,:,:]
	output_img[0] = normalize(output_img[0])
	output_img[1] = normalize(output_img[1])
	# plt.imshow(output_img[0])
	# plt.show()
	# plt.imshow(output_img[1])
	# plt.show()
	# break
	tifffile.imwrite(f'/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_cropped.frame_{frame}.merged_img.tiff', output_img)

(80, 2, 486, 691)


In [93]:
for f in glob.glob('/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_cropped.frame_*.merged_*.tiff'):
	shutil.copyfile(f, os.path.join(train_path, os.path.basename(f)))

# FOV Images

In [73]:
paths = ['/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena','/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena']

In [74]:
def merge_tiffs(brightfield,cy5):
	c1 = normalize(tifffile.imread(brightfield))
	c2 = normalize(tifffile.imread(cy5))
	merged = np.stack([c1,c2],axis=0)
	return merged

In [104]:
for path in paths:
	dirs = glob.glob(os.path.join(path,'*','tif'))
	for d in dirs:
		brightfields = glob.glob(d+'/*.brightfield.tiff')
		cy5s = glob.glob(d+'/*.cy5.tiff')
		brightfields = natsort.natsorted(brightfields)
		cy5 = natsort.natsorted(cy5s)
		
		for b,c in zip(brightfields,cy5):
			if "20241010" in b:
				c1 = normalize(tifffile.imread(b))
				c2 = normalize(tifffile.imread(c))
			elif "20241114" in b:
				c1 = normalize(tifffile.imread(b)[0])
				c2 = normalize(tifffile.imread(c)[0])
			merged = np.stack([c1,c2],axis=0)
			tifffile.imwrite(b.replace('.brightfield.tiff','.merged.tiff'),merged)
			# print('saved',b.replace('.brightfield.tiff','.merged.tiff'))


/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/01.brightfield.tiff /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/01.cy5.tiff
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/02.brightfield.tiff /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/02.cy5.tiff
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/03.brightfield.tiff /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/03.cy5.tiff
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/04.brightfield.tiff /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/04.cy5.tiff
/Users/bebr1814/projects/anabaena/scratc

In [106]:
for path in paths:
	for f in glob.glob(os.path.join(path,'*','tif','*_seg.npy')):
		mask_file = f.replace('_seg.npy','_mask.tiff')
		seg = np.load(f,allow_pickle=True).item()
		mask = seg['masks']
		# plt.imshow(mask)
		# plt.show()
		# break
		tifffile.imwrite(mask_file,mask)
		# print(f)

		dirname = f.split('/')[-3]
		filename = dirname+'.'+os.path.basename(f).replace('brightfield','merged').replace('_seg.npy','_mask.tiff')
		shutil.copyfile(mask_file,os.path.join(train_path,filename))


In [107]:
for path in paths:
	for f in glob.glob(os.path.join(path,'*','tif','*.merged.tiff')):
		dirname = f.split('/')[-3]
		filename = dirname+'.'+os.path.basename(f).replace('merged','merged_img')

		# Conditionally add to training data if the mask exists
		if os.path.exists(os.path.join(train_path,filename).replace('_img','_mask')):
			shutil.copyfile(f,os.path.join(train_path,filename))


# Deprecated

In [8]:
# Testing tiff movies

brightfield = '/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_cropped.tif'
masks = '/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_series1_cellMask_cropped.tif'

outdir = '/Users/bebr1814/projects/anabaena/scratch_data/movies/training_images'


# process brightfield
processor = ImageHandler(
	input_file=brightfield,
	output_dir=outdir,
)
processor.setup_reader()
processor.save_tiff_movie_as_images(
	has_channels=True,
	# selected_frames=[0,10,20,30,40,50,60,70],
	suffix='_img',
)

# process masks
processor = ImageHandler(
	input_file=masks,
	output_dir=outdir,
)
processor.setup_reader()
processor.save_tiff_movie_as_images(
	has_channels=False,
	# selected_frames=[0,10,20,30,40,50,60,70],
	out_prefix=os.path.basename(brightfield).replace('.tif',''),
	suffix='_mask',
)

TIFF Image Shape: (80, 2, 486, 691)
TIFF Data Type: uint16


Frames: 100%|██████████| 80/80 [00:01<00:00, 65.01frame/s]


TIFF Image Shape: (80, 486, 691)
TIFF Data Type: uint8


Frames: 100%|██████████| 80/80 [00:01<00:00, 66.15frame/s]
